In [1]:
import numpy as np
import h5py
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
import os
from sklearn.metrics import f1_score

# ---------------------- 1. 环境配置 ----------------------
def set_chinese_font():
    try:
        font_paths = ['/usr/share/fonts/truetype/wqy/wqy-microhei.ttc',
                      'C:/Windows/Fonts/simhei.ttf',
                      '/System/Library/Fonts/PingFang.ttc']
        for font_path in font_paths:
            if os.path.exists(font_path):
                fm.fontManager.addfont(font_path)
                plt.rcParams['font.family'] = fm.FontProperties(fname=font_path).get_name()
                plt.rcParams['axes.unicode_minus'] = False
                return True
        return False
    except: return False
set_chinese_font()

# ---------------------- 2. FEDformer 核心组件 (保持 Proposed 版本) ----------------------

class FourierBlock_Enhanced(nn.Module):
    def __init__(self, in_channels, out_channels, seq_len, modes=7):
        super(FourierBlock_Enhanced, self).__init__()
        self.modes = modes
        self.scale = (1 / (in_channels * out_channels))
        self.weights = nn.Parameter(self.scale * torch.rand(in_channels, out_channels, self.modes, dtype=torch.cfloat))
        self.bias = nn.Parameter(self.scale * torch.rand(1, self.modes, out_channels, dtype=torch.cfloat))

    def forward(self, x):
        B, L, E = x.shape
        x_ft = torch.fft.rfft(x, dim=1)
        actual_modes = min(self.modes, x_ft.shape[1])
        out_ft = torch.zeros_like(x_ft)
        res = torch.einsum("ble,efl->blf", x_ft[:, :actual_modes, :], self.weights[:, :, :actual_modes])
        out_ft[:, :actual_modes, :] = res + self.bias[:, :actual_modes, :]
        x = torch.fft.irfft(out_ft, n=L, dim=1)
        return x

class FEDformer_EncoderLayer_Enhanced(nn.Module):
    def __init__(self, d_model, nhead, seq_len, modes=7):
        super(FEDformer_EncoderLayer_Enhanced, self).__init__()
        self.feb = FourierBlock_Enhanced(d_model, d_model, seq_len, modes=modes)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(d_model * 4, d_model)
        )
        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        res = x
        x = self.norm1(x)
        x = res + self.dropout(self.feb(x))
        res = x
        x = self.norm2(x)
        x = res + self.dropout(self.ffn(x))
        return x

# ---------------------- 3. 模型定义 (保持 Proposed 结构) ----------------------

class FEDformer_Satellite_Ultimate(nn.Module):
    def __init__(self, input_dim=10, d_model=512, nhead=16, num_layers=4, horizon=10, seq_len=12):
        super(FEDformer_Satellite_Ultimate, self).__init__()
        self.horizon = horizon
        self.conv3 = nn.Conv1d(input_dim, d_model // 4, kernel_size=3, padding=1)
        self.conv5 = nn.Conv1d(input_dim, d_model // 4, kernel_size=5, padding=2)
        self.conv7 = nn.Conv1d(input_dim, d_model // 2, kernel_size=7, padding=3)
        self.bn = nn.BatchNorm1d(d_model)
        self.gelu = nn.GELU()
        
        modes = 7 
        self.pos_emb = nn.Parameter(torch.randn(1, seq_len, d_model) * 0.02)
        self.layers = nn.ModuleList([
            FEDformer_EncoderLayer_Enhanced(d_model, nhead, seq_len, modes=modes) 
            for _ in range(num_layers)
        ])
        
        self.head_horizon = nn.Sequential(
            nn.Linear(d_model, 1024), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(1024, horizon * input_dim), nn.Sigmoid()
        )
        self.head_type = nn.Sequential(
            nn.Linear(d_model * 2, 512), nn.LayerNorm(512), nn.GELU(),
            nn.Dropout(0.4), nn.Linear(512, 5)
        )
        self.head_physics = nn.Sequential(nn.Linear(d_model, 256), nn.GELU(), nn.Linear(256, 2))

    def forward(self, x):
        x_in = x.permute(0, 2, 1)
        x_feat = torch.cat([self.conv3(x_in), self.conv5(x_in), self.conv7(x_in)], dim=1)
        x = self.gelu(self.bn(x_feat)).permute(0, 2, 1)
        x = x + self.pos_emb
        for layer in self.layers:
            x = layer(x)
        avg_feat = torch.mean(x, dim=1); max_feat, _ = torch.max(x, dim=1)
        concat_feat = torch.cat([avg_feat, max_feat], dim=1)
        feat_last = x[:, -1, :] 
        return self.head_horizon(feat_last).view(-1, self.horizon, 10), \
               self.head_type(concat_feat), \
               self.head_physics(avg_feat)

# ---------------------- 4. 训练程序 (消融策略：移除 Scheduler) ----------------------

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0):
        super(FocalLoss, self).__init__()
        self.gamma = gamma; self.ce = nn.CrossEntropyLoss(label_smoothing=0.15)
    def forward(self, input, target):
        logp = self.ce(input, target); p = torch.exp(-logp)
        return ((1 - p) ** self.gamma * logp).mean()

def run_ablation_scheduler_train(data_path, save_pth_path):
    with h5py.File(data_path, 'r') as f:
        X = torch.FloatTensor(f['X'][:]); Y_h = torch.FloatTensor(f['Y_horizon'][:])
        Y_t = torch.LongTensor(f['gt_type'][:])
        P_true = torch.FloatTensor(np.stack([np.mean(Y_h.numpy(), axis=(1,2)), np.sum(Y_h.numpy(), axis=(1,2)) * 0.05], axis=1))

    loader = DataLoader(TensorDataset(X, Y_h, Y_t, P_true), batch_size=32, shuffle=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = FEDformer_Satellite_Ultimate(seq_len=12).to(device)
    
    # 🔥 消融点：移除 OneCycleLR，使用固定的较大学习率，模拟未优化的收敛过程
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=3e-2)
    
    criterion_bce, criterion_focal, criterion_mse = nn.BCELoss(), FocalLoss(), nn.MSELoss()

    print(f"🧪 [Ablation Exp 3] 启动移除 OneCycleLR 策略的对比实验...")
    losses = []
    for epoch in range(60):
        model.train(); total_l = 0
        for bx, byh, byt, bp in loader:
            bx, byh, byt, bp = bx.to(device), byh.to(device), byt.to(device), bp.to(device)
            optimizer.zero_grad()
            ph, pt, pp = model(bx)
            loss = 1.0 * criterion_bce(ph, byh) + 5.0 * criterion_focal(pt, byt) + 0.5 * criterion_mse(pp, bp)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.8)
            optimizer.step()
            # 🔥 消融点：不再调用 scheduler.step()
            total_l += loss.item()
        
        losses.append(total_l/len(loader))
        if (epoch+1) % 10 == 0:
            print(f"Epoch {epoch+1}/60 | 均值 Loss: {losses[-1]:.4f}")

    # 推理评估
    model.eval(); all_h, all_t = [], []
    with torch.no_grad():
        for bx, _, _, _ in DataLoader(TensorDataset(X, Y_h, Y_t, P_true), batch_size=32):
            ph, pt, _ = model(bx.to(device))
            all_h.append(ph.cpu().numpy()); all_t.append(pt.cpu().numpy())
    
    fh, ft = np.concatenate(all_h), np.concatenate(all_t)
    f1 = f1_score(Y_h.numpy().flatten() > 0.5, fh.flatten() > 0.4)
    acc = np.mean(Y_t.numpy() == np.argmax(ft, axis=1))

    print("\n" + "📉" * 15 + "\n【Exp 3: No-Scheduler 消融结果】")
    print(f"🔹 预测态势 F1: {f1:.4f} (预期应低于 0.9803)")
    print(f"🔹 种类识别 Acc: {acc*100:.2f}%")
    print("📉" * 15)

if __name__ == "__main__":
    IN = "/root/autodl-tmp/validate/0218/Prediction/sim_dataset_v6_5/academic_long_horizon_v6_5_200k.h5"
    OUT_PTH = "/root/autodl-tmp/validate/0218/Prediction/FEDformer/ablation_no_scheduler.pth"
    run_ablation_scheduler_train(IN, OUT_PTH)

🧪 [Ablation Exp 3] 启动移除 OneCycleLR 策略的对比实验...
Epoch 10/60 | 均值 Loss: 1.8387
Epoch 20/60 | 均值 Loss: 1.1809
Epoch 30/60 | 均值 Loss: 0.9971
Epoch 40/60 | 均值 Loss: 0.9146
Epoch 50/60 | 均值 Loss: 0.8634
Epoch 60/60 | 均值 Loss: 0.8301

📉📉📉📉📉📉📉📉📉📉📉📉📉📉📉
【Exp 3: No-Scheduler 消融结果】
🔹 预测态势 F1: 0.9737 (预期应低于 0.9803)
🔹 种类识别 Acc: 93.59%
📉📉📉📉📉📉📉📉📉📉📉📉📉📉📉
